# Basic Usage Examples

This notebook provides simple, working examples to get you started with POMDPPlanners quickly.

## Your First POMDP Solution

Let's solve the Tiger POMDP problem step by step:

In [ ]:
from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.core.belief import get_initial_belief

# Step 1: Create the environment
env = TigerPOMDP(discount_factor=0.95)

# Step 2: Create the planner
planner = POMCP(
    environment=env,
    n_simulations=100,  # Reduced for faster testing
    exploration_constant=50.0,
    depth=30
)

# Step 3: Get initial belief
initial_belief = get_initial_belief(env, n_particles=100)  # Reduced for testing

# Step 4: Plan and act
action, run_data = planner.action(initial_belief)
print(f"Recommended action: {action}")
print(f"Planning took {run_data.info_variables['planning_time']:.3f} seconds")

## Running a Complete Episode

Here's how to run a full episode with belief updates:

In [ ]:
from POMDPPlanners.simulations.episodes import run_episode
from POMDPPlanners.utils.logger import get_logger

# Setup logging
logger = get_logger("basic_example")

# Run complete episode
history = run_episode(
    environment=env,
    policy=planner,
    initial_belief=initial_belief,
    num_steps=20,
    logger=logger
)

# Analyze results
print(f"Episode completed in {len(history.history)} steps")
print(f"Total reward: {history.total_reward:.2f}")
print(f"Final state: {history.history[-1].state}")

# Print step-by-step breakdown
for i, step in enumerate(history.history[:5]):  # Show first 5 steps
    print(f"Step {i}: Action={step.action}, Observation={step.observation}, Reward={step.reward}")

## Working with Different Environments

### Continuous Control (CartPole)

In [ ]:
from POMDPPlanners.environments.cartpole_pomdp import CartPolePOMDP
from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
import numpy as np

# Create continuous control environment
env = CartPolePOMDP(
    discount_factor=0.99,
    noise_cov=np.diag([0.1, 0.1, 0.1, 0.1])
)

# Action sampler for continuous actions
class CartPoleActionSampler(ActionSampler):
    def sample(self, belief_node=None):
        return np.random.choice([0, 1])  # Left or right force

# Create planner suitable for continuous domains
planner = PFT_DPW(
    environment=env,
    n_simulations=50,  # Reduced for testing
    action_sampler=CartPoleActionSampler(),
    depth=20
)

# Run episode
belief = get_initial_belief(env, n_particles=50)  # Reduced for testing
action, _ = planner.action(belief)
print(f"CartPole action: {action}")

### Navigation (Light-Dark POMDP)

In [ ]:
from POMDPPlanners.environments.light_dark_pomdp.continuous_light_dark_pomdp import (
    ContinuousLightDarkPOMDP, RewardModelType
)

# Create navigation environment
env = ContinuousLightDarkPOMDP(
    discount_factor=0.95,
    goal_state=np.array([10, 5]),
    start_state=np.array([0, 5]),
    reward_model_type=RewardModelType.STANDARD
)

# Use POMCP for navigation
planner = POMCP(
    environment=env,
    n_simulations=50,  # Reduced for testing
    exploration_constant=10.0
)

# Get action for navigation
belief = get_initial_belief(env, n_particles=50)  # Reduced for testing
action, _ = planner.action(belief)
print(f"Navigation action: {action}")

## Visualization Example

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Collect data during episode
episode_data = {
    'rewards': [],
    'actions': [],
    'observations': [],
    'planning_times': []
}

# Reset to Tiger environment for visualization
env = TigerPOMDP(discount_factor=0.95)
planner = POMCP(
    environment=env,
    n_simulations=50,  # Reduced for testing
    exploration_constant=50.0,
    depth=15  # Reduced for testing
)

# Run episode and collect data
belief = get_initial_belief(env, n_particles=50)  # Reduced for testing
current_state = env.initial_state_dist().sample()[0]

for step in range(10):  # Reduced steps for testing
    action, run_data = planner.action(belief)
    
    # Simulate environment step
    next_state = env.state_transition_model(current_state, action).sample()[0]
    observation = env.observation_model(next_state, action).sample()[0]
    reward = env.reward(current_state, action)
    
    # Update belief (simplified)
    belief = planner.update_belief(belief, action, observation)
    current_state = next_state
    
    # Store data
    episode_data['rewards'].append(reward)
    episode_data['actions'].append(action)
    episode_data['observations'].append(observation)
    episode_data['planning_times'].append(run_data.info_variables.get('planning_time', 0))
    
    if env.is_terminal(current_state):
        break

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot rewards over time
axes[0, 0].plot(episode_data['rewards'])
axes[0, 0].set_title('Rewards per Step')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Reward')

# Plot planning times
axes[0, 1].plot(episode_data['planning_times'])
axes[0, 1].set_title('Planning Time per Step')
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('Time (seconds)')

# Plot cumulative reward
cumulative_rewards = np.cumsum(episode_data['rewards'])
axes[1, 0].plot(cumulative_rewards)
axes[1, 0].set_title('Cumulative Reward')
axes[1, 0].set_xlabel('Step')
axes[1, 0].set_ylabel('Cumulative Reward')

# Action frequency
action_counts = Counter(episode_data['actions'])
axes[1, 1].bar(range(len(action_counts)), list(action_counts.values()))
axes[1, 1].set_title('Action Frequency')
axes[1, 1].set_xlabel('Action')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_xticks(range(len(action_counts)))
axes[1, 1].set_xticklabels(list(action_counts.keys()))

plt.tight_layout()
plt.show()

print(f"\nEpisode completed with {len(episode_data['rewards'])} steps")
print(f"Total reward: {sum(episode_data['rewards']):.2f}")
print(f"Average planning time: {np.mean(episode_data['planning_times']):.3f} seconds")

## Quick Configuration Tips

**Performance Tuning**
- Start with `n_simulations=100` for quick testing
- Increase to `1000-2000` for better performance
- Use `exploration_constant=50.0` for Tiger POMDP
- Adjust `depth` based on problem horizon

**Common Issues**
- Low rewards? Increase `n_simulations`
- Slow planning? Decrease `depth` or `n_simulations`
- Poor exploration? Adjust `exploration_constant`

**Memory Usage**
- Use fewer particles (`n_particles=500`) for large state spaces
- Reduce `depth` for memory-constrained environments

## Next Steps

- Try [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for optimization examples
- See [planners_comparison.ipynb](planners_comparison.ipynb) for algorithm comparisons
- Check the API documentation for detailed reference